In [3]:
from pathlib import Path
import os
import sys
import json

# Keep the notebook usable when launched from the repository root or this directory.
WORKBENCH_ROOT = Path("/project_ghent/ADAPT/adapt-workbench")
INFERENCE_ROOT = Path("/project_ghent/ADAPT/boltz_inference")
ADAPT_BOLTZGEN_ROOT = Path("/project_ghent/ADAPT/adapt-boltzgen")
BENCHMARK_ROOT = Path("/project_ghent/ADAPT/benchmark-design")

for path in [
    WORKBENCH_ROOT / "src",
    INFERENCE_ROOT / "src",
    ADAPT_BOLTZGEN_ROOT / "src",
    ADAPT_BOLTZGEN_ROOT / "boltzgen_adapt" / "src",
    ADAPT_BOLTZGEN_ROOT / "boltzgen" / "src",
    BENCHMARK_ROOT / "src",
]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

from adapt_workbench.dataset import load_task_dataset
from adapt_workbench.design.adapt_design import to_backbone

DATASET_PATH = WORKBENCH_ROOT / "data" / "rabd_v2_old"
MODEL_CONFIG = WORKBENCH_ROOT / "config" / "designbg_87-pretrained.yaml"
NUM_SAMPLES = 1

print(f"Dataset: {DATASET_PATH}")
print(f"Model config: {MODEL_CONFIG}")
print(f"Torch/CUDA will be used by the configured inference implementation.")

Hostname: 4a9e1194cde4
Working directory: /


In [ ]:
tasks = load_task_dataset(DATASET_PATH)
assert tasks, f"No tasks found in {DATASET_PATH}"

TASK_INDEX = 0
TASK = tasks[TASK_INDEX]
TASK.item["hotspot"] = []  # Match the backbone CLI, which disables hotspot conditioning.

print(f"Loaded {len(tasks)} tasks; using task {TASK_INDEX}: {TASK.task_id}\n")
print("Reference PDB:", TASK.pdb_path)
print("Item keys:", sorted(TASK.item))
print("Binder chains:", TASK.binder_ids)
print("Target chains:", TASK.target_ids)
TASK.item

# Inspect Backbone Generation

This notebook follows one `rabd_v2_old` example through the backbone-generation pipeline. It keeps the intermediate objects visible, with special attention to the dictionary returned by the sampling step.

## Prompt

> For one task from `rabd_v2_old`, inspect the complete backbone-generation path: load the task and reference structure, initialize the configured design model, build the model input/features, run exactly one `sample_new` diffusion sample, expose the raw `prediction` dictionary and its tensor shapes, convert the generated output to PDB, and visualize the final backbone. Keep expensive work in explicit cells so each stage can be inspected independently.